# CWSB — full end-to-end retrain pipeline (one-run Colab)

**Workflow:** upload three zips → set the version number → Run all → download
`cwsb_v{N}.pt` + `cwsb_v{N}.onnx` (with thresholds, parity check and an old-vs-new comparison).

**The three zips** (top level of Google Drive, or browser upload):

| Zip | Contents | Required? |
|---|---|---|
| `old_data.zip` | a `healthy` folder + an `unhealthy` folder of the previous photos | **yes** |
| `new_data.zip` | `healthy.zip` + `unhealthy.zip` (or plain folders) of the new photos | **yes** |
| `old_models.zip` | the previous `cwsb_vN.pt` (+ `.onnx`) | optional — used **only** to compare old vs new on the same test set; training never starts from it |

Folder names inside the zips are matched loosely: anything containing *unhealthy* or *infect*
counts as Unhealthy; anything else containing *health* counts as Healthy.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save (do this first — changing
it later wipes uploaded files). Total time ≈ 1–1.5 h depending on data size.

Pipeline stages (the project's own scripts, embedded unchanged):
clean → plant-level split (leak-proof) → healthy tiling → ConvNeXt train → thresholds from the
**healthy** validation scores → test-set evaluation (new vs old model) → ONNX export + 4-decimal parity.

In [ ]:
# Cell 1 - SET THE VERSION, check GPU, install
VERSION_NUMBER = 2          # <<< fill in manually: outputs become cwsb_v2.pt / cwsb_v2.onnx etc.

# Strip the background from every photo before training, and record that in
# config.json so serving does exactly the same thing. ONE switch, both sides.
#   False -> behaves exactly as before (v1 behaviour, full frames)
#   True  -> ONLY for datasets that are all STEM CLOSE-UPS. On whole-plant
#            canopy photos the stem cannot be separated (measured: fails on
#            ~80%) and those photos pass through untouched, which would give
#            you a half-processed dataset - worse than not doing it at all.
REMOVE_BACKGROUND = False
BG_FILL = 'blur'            # 'blur' | 'black' | 'grey'

!nvidia-smi -L || echo 'NO GPU - Runtime > Change runtime type > T4 GPU (fix NOW, it wipes files)'
%pip -q install timm onnx onnxruntime onnxscript
import torch
TAG = f'cwsb_v{VERSION_NUMBER}'
print('version tag:', TAG, '| torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# Cell 2 - fetch the three zips and merge old + new photos into Data_final/
import os, shutil, zipfile
from pathlib import Path

BASE = Path('/content/cwsb')
DATA = BASE / 'Data_final'
H, U = DATA / 'Healthy', DATA / 'Unhealthy(Infected)'
shutil.rmtree(BASE, ignore_errors=True)
for d in (H, U, BASE / 'src'):
    d.mkdir(parents=True)

USE_DRIVE = True   # False = browser upload instead
wanted = ['old_data.zip', 'new_data.zip', 'old_models.zip']
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    for n in wanted:
        src = Path('/content/drive/MyDrive') / n
        if src.exists():
            shutil.copy(src, '/content/' + n)
else:
    from google.colab import files
    print('upload old_data.zip, new_data.zip and (optional) old_models.zip:')
    files.upload()

for n in ('old_data.zip', 'new_data.zip'):
    assert Path('/content/' + n).exists(), f'{n} is missing - it is required'

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
copied = {'Healthy': 0, 'Unhealthy': 0}
skipped = []

def route(rel_path):
    s = rel_path.lower()
    if 'unhealthy' in s or 'infect' in s:
        return U
    if 'health' in s:
        return H
    return None

def ingest(root, tag):
    root = Path(root)
    for p in sorted(root.rglob('*')):
        if p.suffix.lower() not in IMG_EXTS or p.name.startswith('._') or '__MACOSX' in p.parts:
            continue
        dest_dir = route(str(p.relative_to(root)))
        if dest_dir is None:
            skipped.append(str(p.relative_to(root)))
            continue
        dest = dest_dir / (p.stem + '.jpg')          # clean.py reads *.jpg (case-sensitive on Linux)
        if dest.exists():
            dest = dest_dir / f'{tag}_{p.stem}.jpg'  # avoid old/new name collisions
        shutil.copy(p, dest)
        copied['Healthy' if dest_dir is H else 'Unhealthy'] += 1

def unzip(src, dst):
    with zipfile.ZipFile(src) as zf:
        zf.extractall(dst)

unzip('/content/old_data.zip', '/content/_old')
ingest('/content/_old', 'old')

unzip('/content/new_data.zip', '/content/_new')
for inner in sorted(Path('/content/_new').rglob('*.zip')):   # healthy.zip / unhealthy.zip inside
    unzip(inner, inner.with_suffix(''))
    inner.unlink()
ingest('/content/_new', 'new')

OLD_PT = None
if Path('/content/old_models.zip').exists():
    unzip('/content/old_models.zip', '/content/_oldmodels')
    pts = sorted(Path('/content/_oldmodels').rglob('*.pt'))
    OLD_PT = str(pts[0]) if pts else None
print('old model for comparison:', OLD_PT or 'none provided')

print(f"merged -> Healthy {copied['Healthy']}  Unhealthy {copied['Unhealthy']}")
if skipped:
    print(f'WARNING: {len(skipped)} images skipped (folder name says neither healthy nor unhealthy):')
    for s in skipped[:10]:
        print('  ', s)
assert copied['Healthy'] > 0 and copied['Unhealthy'] > 0, 'one class has no images - check zip layout'

In [ ]:
# Cell - project script, embedded verbatim
%%writefile /content/cwsb/src/clean.py
"""Phase 1a - Clean Data_final into work/clean.

Per image: apply EXIF rotation, strip ALL metadata (GPS!), save plain JPEG q95
at full resolution, and record quality scores (blur / clipped / dark), SHA-256
and a perceptual dhash. Then report exact and near duplicates.

Data_final/ is read-only; everything is written to work/clean/.
"""
import csv
import hashlib
import sys
from pathlib import Path

import cv2
import numpy as np
from PIL import Image, ImageOps

ROOT = Path(__file__).resolve().parents[1]
SRC = ROOT / "Data_final"
DST = ROOT / "work" / "clean"
CLASSES = {"Healthy": "Healthy", "Unhealthy(Infected)": "Unhealthy"}

# Quality flags (tuned for big phone photos)
BLUR_MIN = 60.0        # Laplacian variance below this => blurry
CLIPPED_MAX = 0.20     # >20% near-white pixels => burnt out (sun)
DARK_MAX = 0.40        # >40% near-black pixels => too dark
NEAR_DUP_HAMMING = 6   # dhash distance <= this => near duplicate


def dhash(gray: np.ndarray, size: int = 8) -> int:
    small = cv2.resize(gray, (size + 1, size), interpolation=cv2.INTER_AREA)
    bits = small[:, 1:] > small[:, :-1]
    return int("".join("1" if b else "0" for b in bits.flatten()), 2)


def hamming(a: int, b: int) -> int:
    return bin(a ^ b).count("1")


def main() -> int:
    rows = []
    for src_cls, dst_cls in CLASSES.items():
        out_dir = DST / dst_cls
        out_dir.mkdir(parents=True, exist_ok=True)
        files = sorted((SRC / src_cls).glob("*.jpg"))
        print(f"{src_cls}: {len(files)} images")
        for i, p in enumerate(files):
            img = ImageOps.exif_transpose(Image.open(p)).convert("RGB")
            out = out_dir / p.name
            img.save(out, "JPEG", quality=95, exif=b"")  # strips GPS/EXIF

            arr = np.asarray(img)
            g_full = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
            scale = 1024 / max(g_full.shape[1], 1)
            g = cv2.resize(g_full, (1024, max(int(g_full.shape[0] * scale), 1)))
            blur = float(cv2.Laplacian(g, cv2.CV_64F).var())
            clipped = float((g > 250).mean())
            dark = float((g < 15).mean())

            flags = []
            if blur < BLUR_MIN:
                flags.append("blurry")
            if clipped > CLIPPED_MAX:
                flags.append("clipped")
            if dark > DARK_MAX:
                flags.append("dark")

            rows.append({
                "file": f"{dst_cls}/{p.name}",
                "label": dst_cls,
                "width": img.width,
                "height": img.height,
                "blur": round(blur, 2),
                "clipped": round(clipped, 4),
                "dark": round(dark, 4),
                "sha256": hashlib.sha256(out.read_bytes()).hexdigest(),
                "dhash": f"{dhash(g):016x}",
                "flags": "|".join(flags),
            })
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(files)}")

    with open(DST / "images.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)

    # Duplicate report
    dupes = []
    for i in range(len(rows)):
        for j in range(i + 1, len(rows)):
            a, b = rows[i], rows[j]
            if a["sha256"] == b["sha256"]:
                dupes.append((a["file"], b["file"], "exact", 0))
            else:
                d = hamming(int(a["dhash"], 16), int(b["dhash"], 16))
                if d <= NEAR_DUP_HAMMING:
                    dupes.append((a["file"], b["file"], "near", d))
    with open(DST / "duplicates.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["file_a", "file_b", "kind", "hamming"])
        w.writerows(dupes)

    n_flag = sum(1 for r in rows if r["flags"])
    cross = [d for d in dupes if d[0].split("/")[0] != d[1].split("/")[0]]
    print(f"\nDone: {len(rows)} images cleaned -> {DST}")
    print(f"Flagged (excluded from training): {n_flag}")
    for r in rows:
        if r["flags"]:
            print(f"  {r['file']}: {r['flags']} (blur={r['blur']}, clipped={r['clipped']}, dark={r['dark']})")
    print(f"Duplicate pairs: {len(dupes)} ({len(cross)} CROSS-CLASS - label errors, must review)")
    for d in dupes:
        print(f"  {d[2]} d={d[3]}: {d[0]} <-> {d[1]}")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
# Cell - project script, embedded verbatim
%%writefile /content/cwsb/src/make_split.py
"""Phase 1b - Group photos into pseudo-plants and make a leak-proof split.

Consecutive frame numbers (gap <= 4) or timestamps within 120 s from the same
camera series are treated as ONE plant. The split is done on plants, not
photos: 70/15/15 per class, so no plant ever appears in two sets. Flagged
(blurry/clipped/dark) images are excluded. The script refuses to finish if
any leakage is detected.
"""
import csv
import random
import re
import sys
from collections import defaultdict
from datetime import datetime
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
CLEAN = ROOT / "work" / "clean"
OUT = ROOT / "work" / "splits" / "split.csv"

FRAME_GAP = 4       # frames
TIME_GAP = 120.0    # seconds
FRACTIONS = {"train": 0.70, "val": 0.15, "test": 0.15}
SEED = 42

TS_RE = re.compile(r"(?P<pre>.*IMG)_(?P<d>\d{8})_(?P<t>\d{6})", re.I)
NUM_RE = re.compile(r"^(?P<pre>.*?)(?P<num>\d+)(?P<post>[^\d]*)$")


def sort_key(name, label):
    stem = Path(name).stem
    m = TS_RE.match(stem)
    if m:
        ts = datetime.strptime(m.group("d") + m.group("t"), "%Y%m%d%H%M%S")
        return (label, "ts:" + m.group("pre"), ts.timestamp())
    m = NUM_RE.match(stem)
    if m:
        return (label, "fr:" + m.group("pre") + "|" + m.group("post"), float(m.group("num")))
    return (label, "one:" + stem, 0.0)


def main() -> int:
    with open(CLEAN / "images.csv", newline="") as f:
        rows = list(csv.DictReader(f))

    # --- group into pseudo-plants (within class + camera series only) ---
    keyed = sorted(rows, key=lambda r: sort_key(r["file"], r["label"]))
    groups = []
    prev = None
    for r in keyed:
        label, series, pos = sort_key(r["file"], r["label"])
        gap = TIME_GAP if series.startswith("ts:") else FRAME_GAP
        if prev and prev[0] == (label, series) and pos - prev[1] <= gap:
            groups[-1].append(r)
        else:
            groups.append([r])
        prev = ((label, series), pos)
    for gi, g in enumerate(groups):
        for r in g:
            r["group"] = f"plant_{gi:03d}"

    # --- split groups per class, excluding flagged images from counts ---
    rng = random.Random(SEED)
    by_class = defaultdict(list)
    for g in groups:
        usable = [r for r in g if not r["flags"]]
        if usable:
            by_class[g[0]["label"]].append(g)

    assign = {}
    for label, gs in by_class.items():
        rng.shuffle(gs)
        total = sum(len([r for r in g if not r["flags"]]) for g in gs)
        want = {s: total * frac for s, frac in FRACTIONS.items()}
        got = {s: 0 for s in FRACTIONS}
        # give each split one group first (so every set has both classes)
        order = ["test", "val", "train"]
        for s, g in zip(order, gs):
            assign[g[0]["group"]] = s
            got[s] += len([r for r in g if not r["flags"]])
        for g in gs[len(order):]:
            s = max(FRACTIONS, key=lambda s: want[s] - got[s])
            assign[g[0]["group"]] = s
            got[s] += len([r for r in g if not r["flags"]])

    for r in rows:
        r["split"] = "excluded" if r["flags"] else assign.get(r["group"], "excluded")

    # --- leakage checks ---
    plant_splits = defaultdict(set)
    for r in rows:
        if r["split"] != "excluded":
            plant_splits[r["group"]].add(r["split"])
    leaks = {g: s for g, s in plant_splits.items() if len(s) > 1}
    if leaks:
        print(f"LEAKAGE: {leaks}")
        return 1

    dup_leaks = []
    dup_file = CLEAN / "duplicates.csv"
    if dup_file.exists():
        split_of = {r["file"]: r["split"] for r in rows}
        with open(dup_file, newline="") as f:
            for d in csv.DictReader(f):
                sa, sb = split_of.get(d["file_a"]), split_of.get(d["file_b"])
                if sa and sb and sa != sb and "excluded" not in (sa, sb):
                    dup_leaks.append((d["file_a"], sa, d["file_b"], sb, d["kind"]))
    for d in dup_leaks:
        print(f"WARNING near-dup across splits: {d}")

    OUT.parent.mkdir(parents=True, exist_ok=True)
    with open(OUT, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["file", "label", "group", "split"])
        w.writeheader()
        w.writerows([{k: r[k] for k in ("file", "label", "group", "split")} for r in rows])

    print(f"Plants: {len(groups)} total "
          f"(Healthy {len(by_class.get('Healthy', []))}, Unhealthy {len(by_class.get('Unhealthy', []))})")
    tally = defaultdict(lambda: defaultdict(int))
    for r in rows:
        tally[r["split"]][r["label"]] += 1
    for s in ("train", "val", "test", "excluded"):
        h, u = tally[s].get("Healthy", 0), tally[s].get("Unhealthy", 0)
        print(f"  {s:8s}: {h:3d} healthy + {u:3d} unhealthy = {h + u}")
    print("LEAKAGE CHECK OK" + (f" ({len(dup_leaks)} near-dup warnings)" if dup_leaks else ""))
    print(f"-> {OUT}")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
# Cell - project script, embedded verbatim
%%writefile /content/cwsb/src/make_tiles.py
"""Phase 2a - Multi-crop tiling for the tiny healthy class.

TRAIN-split healthy photos only (never val/test): each big photo (3000-6000 px)
is cut into overlapping square tiles covering different stem/canopy regions -
genuinely different pixel regions, not synthetic copies. 28 photos -> ~170
extra real samples. Tiles inherit their photo's plant group, so no leakage.
"""
import csv
import sys
from pathlib import Path

from PIL import Image

ROOT = Path(__file__).resolve().parents[1]
CLEAN = ROOT / "work" / "clean"
SPLIT = ROOT / "work" / "splits" / "split.csv"
OUT = ROOT / "work" / "tiles" / "Healthy"

GRID = (3, 2)  # cols x rows of overlapping tiles + 1 center tile


def main() -> int:
    OUT.mkdir(parents=True, exist_ok=True)
    with open(SPLIT, newline="") as f:
        rows = [r for r in csv.DictReader(f)
                if r["label"] == "Healthy" and r["split"] == "train"]

    manifest = []
    for r in rows:
        img = Image.open(CLEAN / r["file"])
        w, h = img.size
        side = int(min(w, h) * 0.55)  # tile side: 55% of short edge -> overlap
        cols, rows_n = GRID
        xs = [int(i * (w - side) / max(cols - 1, 1)) for i in range(cols)]
        ys = [int(j * (h - side) / max(rows_n - 1, 1)) for j in range(rows_n)]
        boxes = [(x, y) for y in ys for x in xs]
        boxes.append(((w - side) // 2, (h - side) // 2))  # center
        stem = Path(r["file"]).stem
        for k, (x, y) in enumerate(boxes):
            name = f"{stem}_tile{k}.jpg"
            img.crop((x, y, x + side, y + side)).save(OUT / name, "JPEG", quality=95)
            manifest.append({"file": f"Healthy/{name}", "label": "Healthy",
                             "group": r["group"], "split": "train"})

    with open(ROOT / "work" / "tiles" / "tiles.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["file", "label", "group", "split"])
        w.writeheader()
        w.writerows(manifest)
    print(f"{len(rows)} train healthy photos -> {len(manifest)} tiles in {OUT}")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
# Cell - project script, embedded verbatim
%%writefile /content/cwsb/src/train.py
"""Phase 2b - Train the CWSB classifier. Runs on Colab GPU (or CPU, slowly).

Two contenders, validation decides which ships:
  --mode A : frozen ConvNeXt-Tiny backbone -> logistic regression head
             (~1.5k trainable params, nearly impossible to overfit 28 images)
  --mode B : full ConvNeXt-Tiny fine-tune (2-epoch head warm-up, then all
             layers), weighted sampling + class-weighted loss + augmentation

Positive class = Unhealthy (infected). Model selection metric = validation
PR-AUC (average precision). Seed 42 everywhere.

Usage (from the project root, after clean/split/tiles):
  python src/train.py --mode A
  python src/train.py --mode B
"""
import argparse
import csv
import json
import random
import time
from pathlib import Path

import numpy as np
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms

ROOT = Path(__file__).resolve().parents[1]
CLEAN = ROOT / "work" / "clean"
TILES = ROOT / "work" / "tiles"
SPLIT = ROOT / "work" / "splits" / "split.csv"
TILES_CSV = ROOT / "work" / "tiles" / "tiles.csv"
OUT = ROOT / "models" / "v1"

IMG = 384
BACKBONE = "convnext_tiny.fb_in22k_ft_in1k_384"
SEED = 42
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.10),
    transforms.RandomApply([transforms.GaussianBlur(5)], p=0.15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([  # MUST stay identical in the API
    transforms.Resize(IMG + 32),
    transforms.CenterCrop(IMG),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class CwsbDataset(Dataset):
    def __init__(self, rows, tf):
        self.rows, self.tf = rows, tf

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        base = TILES if r.get("tile") else CLEAN
        img = Image.open(base / r["file"]).convert("RGB")
        return self.tf(img), 1 if r["label"] == "Unhealthy" else 0


def load_rows(split, with_tiles=False):
    with open(SPLIT, newline="") as f:
        rows = [r for r in csv.DictReader(f) if r["split"] == split]
    if with_tiles and TILES_CSV.exists():
        with open(TILES_CSV, newline="") as f:
            rows += [dict(r, tile="1") for r in csv.DictReader(f) if r["split"] == split]
    return rows


def set_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


@torch.no_grad()
def scores(model, rows, device, bs=16):
    model.eval()
    dl = DataLoader(CwsbDataset(rows, eval_tf), batch_size=bs, num_workers=2)
    ys, ps = [], []
    for x, y in dl:
        p = torch.softmax(model(x.to(device)), 1)[:, 1]
        ps += p.cpu().tolist()
        ys += y.tolist()
    return np.array(ys), np.array(ps)


@torch.no_grad()
def features(backbone, rows, device, bs=16):
    backbone.eval()
    dl = DataLoader(CwsbDataset(rows, eval_tf), batch_size=bs, num_workers=2)
    feats, ys = [], []
    for x, y in dl:
        f = backbone(x.to(device))
        feats.append(f.cpu().numpy())
        ys += y.tolist()
    return np.concatenate(feats), np.array(ys)


def run_a(device):
    """Frozen backbone + logistic regression."""
    backbone = timm.create_model(BACKBONE, pretrained=True, num_classes=0).to(device)
    tr = load_rows("train", with_tiles=True)
    va = load_rows("val")
    print(f"A: {len(tr)} train rows (incl. tiles), {len(va)} val")
    t0 = time.time()
    Xtr, ytr = features(backbone, tr, device)
    Xva, yva = features(backbone, va, device)
    clf = LogisticRegression(class_weight="balanced", max_iter=5000, C=0.1)
    clf.fit(Xtr, ytr)
    ap = average_precision_score(yva, clf.predict_proba(Xva)[:, 1])
    print(f"A: val PR-AUC = {ap:.4f}  ({time.time() - t0:.0f}s)")
    OUT.mkdir(parents=True, exist_ok=True)
    torch.save({"mode": "A", "backbone": BACKBONE, "img": IMG,
                "coef": clf.coef_, "intercept": clf.intercept_,
                "val_pr_auc": ap}, OUT / "contender_A.pt")
    return ap


def run_b(device, epochs=40, patience=6):
    """Full fine-tune with warm-up, weighted sampling, class-weighted loss."""
    model = timm.create_model(BACKBONE, pretrained=True, num_classes=2).to(device)
    tr = load_rows("train", with_tiles=True)
    va = load_rows("val")
    n_pos = sum(r["label"] == "Unhealthy" for r in tr)
    n_neg = len(tr) - n_pos
    print(f"B: train {len(tr)} ({n_neg} healthy incl. tiles, {n_pos} unhealthy), val {len(va)}")

    weights = [1.0 / (n_pos if r["label"] == "Unhealthy" else n_neg) for r in tr]
    sampler = WeightedRandomSampler(weights, num_samples=len(tr), replacement=True)
    dl = DataLoader(CwsbDataset(tr, train_tf), batch_size=16, sampler=sampler, num_workers=2)
    cw = torch.tensor([len(tr) / (2 * n_neg), len(tr) / (2 * n_pos)], dtype=torch.float32)
    crit = nn.CrossEntropyLoss(weight=cw.to(device))

    head = [p for n, p in model.named_parameters() if "head" in n]
    body = [p for n, p in model.named_parameters() if "head" not in n]

    best_ap, best_ep, since = 0.0, -1, 0
    OUT.mkdir(parents=True, exist_ok=True)
    for ep in range(epochs):
        warm = ep < 2
        for p in body:
            p.requires_grad = not warm
        opt = torch.optim.AdamW(
            [{"params": head, "lr": 1e-3 if warm else 1e-4},
             {"params": body, "lr": 0 if warm else 1e-4}], weight_decay=0.05)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=len(dl))
        model.train()
        tot = 0.0
        for x, y in dl:
            opt.zero_grad()
            loss = crit(model(x.to(device)), y.to(device))
            loss.backward()
            opt.step()
            sched.step()
            tot += loss.item()
        yva, pva = scores(model, va, device)
        ap = average_precision_score(yva, pva)
        print(f"B ep{ep:02d} {'warm' if warm else 'full'}: loss {tot / len(dl):.4f}  val PR-AUC {ap:.4f}")
        if ap > best_ap:
            best_ap, best_ep, since = ap, ep, 0
            torch.save({"mode": "B", "backbone": BACKBONE, "img": IMG,
                        "state_dict": model.state_dict(),
                        "val_pr_auc": ap, "epoch": ep}, OUT / "contender_B.pt")
        else:
            since += 1
            if since >= patience:
                print(f"B: early stop (best ep{best_ep}, PR-AUC {best_ap:.4f})")
                break
    return best_ap


def main():
    ap_ = argparse.ArgumentParser()
    ap_.add_argument("--mode", choices=["A", "B", "both"], default="both")
    args = ap_.parse_args()
    set_seed()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device: {device}")

    results = {}
    if args.mode in ("A", "both"):
        results["A"] = run_a(device)
    if args.mode in ("B", "both"):
        results["B"] = run_b(device)
    if results:
        winner = max(results, key=results.get)
        (OUT / "training_result.json").write_text(json.dumps(
            {"val_pr_auc": results, "winner": winner,
             "backbone": BACKBONE, "img": IMG, "seed": SEED}, indent=2))
        print(f"\nWinner on validation: contender {winner}  {results}")


if __name__ == "__main__":
    main()


In [ ]:
# Cell - project script, embedded verbatim
%%writefile /content/cwsb/src/remove_bg.py
"""Label-free background removal for stem close-ups.

No annotations, no extra model download: OpenCV GrabCut seeded with a centre
rectangle, which works because the capture protocol puts the stem in the
middle of the frame.

THE RULE: whatever this does to an image must happen in BOTH training and
serving. The model is trained on the output of `apply()`, so the API must
call the same `apply()` before predicting. Masking only at prediction time
makes things worse - measured, see reports/mask_experiment_images_16_11.csv.

It is deliberately conservative: when GrabCut collapses (finds nothing) or
keeps nearly everything, the mask is rejected and the original image is
returned unchanged, with `ok=False`. Whole-plant canopy photos fail this way
about 80% of the time - there is no figure/ground to separate - so this is
for stem close-ups only.

  python src/remove_bg.py images_16_11 --out work/nobg/images_16_11
  python src/remove_bg.py images_16_11 --out work/nobg/x --fill blur --preview
"""
import argparse
from pathlib import Path

import cv2
import numpy as np
from PIL import Image, ImageOps

ROOT = Path(__file__).resolve().parents[1]
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

WORK = 480          # GrabCut runs at this long side, then the mask is upscaled
MARGIN = 0.12       # seed rectangle inset from each edge
ITERS = 5
KEEP_MIN, KEEP_MAX = 0.08, 0.85   # outside this the mask is not trusted
FEATHER = 7         # soft edge, avoids a hard cut-out look


def mask_for(rgb):
    """Foreground mask (float 0..1, image size). Uses no labels."""
    h, w = rgb.shape[:2]
    s = WORK / max(h, w)
    small = cv2.resize(rgb, (max(int(w * s), 1), max(int(h * s), 1)))
    sh, sw = small.shape[:2]
    rect = (int(sw * MARGIN), int(sh * MARGIN),
            int(sw * (1 - 2 * MARGIN)), int(sh * (1 - 2 * MARGIN)))

    m = np.zeros((sh, sw), np.uint8)
    bgd, fgd = np.zeros((1, 65), np.float64), np.zeros((1, 65), np.float64)
    try:
        cv2.grabCut(cv2.cvtColor(small, cv2.COLOR_RGB2BGR), m, rect,
                    bgd, fgd, ITERS, cv2.GC_INIT_WITH_RECT)
    except cv2.error:
        return None
    fg = np.where((m == cv2.GC_FGD) | (m == cv2.GC_PR_FGD), 1.0, 0.0).astype(np.float32)

    # keep only the largest blob: drops specks of stray foreground
    n, lab, stats, _ = cv2.connectedComponentsWithStats((fg > 0.5).astype(np.uint8), 8)
    if n > 1:
        biggest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        fg = (lab == biggest).astype(np.float32)

    fg = cv2.GaussianBlur(fg, (FEATHER, FEATHER), 0)
    return cv2.resize(fg, (w, h), interpolation=cv2.INTER_LINEAR)


def apply(pil, fill="black"):
    """(PIL image, ok, kept_fraction). ok=False -> original returned unchanged.

    fill: 'black' | 'grey' | 'blur'  - what replaces the background.
          'blur' keeps the scene's colours but destroys its detail, which is
          gentler than a hard cut-out and closer to what the camera sees.
    """
    rgb = np.asarray(pil.convert("RGB"))
    m = mask_for(rgb)
    if m is None:
        return pil, False, 0.0
    kept = float((m > 0.5).mean())
    if not (KEEP_MIN < kept < KEEP_MAX):
        return pil, False, kept            # untrustworthy mask: change nothing

    if fill == "blur":
        k = max(int(min(rgb.shape[:2]) * 0.05) | 1, 9)
        back = cv2.GaussianBlur(rgb, (k, k), 0)
    elif fill == "grey":
        back = np.full_like(rgb, 128)
    else:
        back = np.zeros_like(rgb)

    out = rgb * m[..., None] + back * (1.0 - m[..., None])
    return Image.fromarray(out.astype(np.uint8)), True, kept


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("folder")
    ap.add_argument("--out", required=True, help="where the processed copies go")
    ap.add_argument("--fill", choices=["black", "grey", "blur"], default="black")
    ap.add_argument("--preview", action="store_true",
                    help="also write a before/after contact sheet PDF")
    args = ap.parse_args()

    src = Path(args.folder)
    src = src if src.is_absolute() else ROOT / src
    dst = Path(args.out)
    dst = dst if dst.is_absolute() else ROOT / dst
    dst.mkdir(parents=True, exist_ok=True)

    paths = sorted(p for p in src.rglob("*") if p.suffix.lower() in EXTS)
    print(f"{len(paths)} images in {src}  ->  {dst}   (fill: {args.fill})")

    done, skipped, keeps, shots = 0, [], [], []
    for p in paths:
        pil = ImageOps.exif_transpose(Image.open(p)).convert("RGB")
        out, ok, kept = apply(pil, args.fill)
        out.save(dst / (p.stem + ".jpg"), quality=95)
        if ok:
            done += 1
            keeps.append(kept)
            if args.preview and len(shots) < 24:
                shots.append((p.name, np.asarray(pil.copy().resize((320, 320))),
                              np.asarray(out.copy().resize((320, 320))), kept))
        else:
            skipped.append((p.name, kept))

    print(f"\nbackground removed : {done}/{len(paths)}")
    print(f"left unchanged     : {len(skipped)}/{len(paths)}  (mask not trustworthy)")
    if keeps:
        print(f"kept area          : median {np.median(keeps):.0%}  "
              f"range {min(keeps):.0%}-{max(keeps):.0%}")
    for name, kept in skipped[:10]:
        print(f"  unchanged: {name}  (grabcut kept {kept:.0%})")

    if args.preview and shots:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        from matplotlib.backends.backend_pdf import PdfPages
        pdf_path = dst / "_preview.pdf"
        with PdfPages(pdf_path) as pdf:
            for i in range(0, len(shots), 3):
                chunk = shots[i:i + 3]
                fig, axes = plt.subplots(len(chunk), 2, figsize=(7, 3.4 * len(chunk)),
                                         squeeze=False)
                for r, (name, before, after, kept) in enumerate(chunk):
                    axes[r][0].imshow(before); axes[r][0].set_title(name, fontsize=8)
                    axes[r][1].imshow(after)
                    axes[r][1].set_title(f"background removed (kept {kept:.0%})", fontsize=8)
                    for a in axes[r]:
                        a.axis("off")
                fig.tight_layout(); pdf.savefig(fig, dpi=85); plt.close(fig)
        print(f"-> {pdf_path}")


if __name__ == "__main__":
    main()


In [ ]:
# Cell 3 - run the data pipeline: clean -> split (leak-proof) -> tiles
import subprocess, sys
os.chdir('/content/cwsb')
for s in ['src/clean.py', 'src/make_split.py', 'src/make_tiles.py']:
    if s == 'src/make_split.py' and REMOVE_BACKGROUND:
        # work/clean is the single source for split, tiles AND train, so
        # processing it here means every later stage sees the same pixels
        print('\n' + '=' * 25, 'background removal', '=' * 25)
        sys.path.insert(0, '/content/cwsb/src')
        import remove_bg
        from PIL import Image, ImageOps
        done = unchanged = 0
        for p in sorted(Path('work/clean').rglob('*.jpg')):
            pil = ImageOps.exif_transpose(Image.open(p)).convert('RGB')
            out, ok, _ = remove_bg.apply(pil, BG_FILL)
            out.save(p, quality=95)
            done, unchanged = done + bool(ok), unchanged + (not ok)
        print(f'background removed from {done} photos; {unchanged} left unchanged')
        if unchanged > done * 0.25:
            print('WARNING: many photos could not be processed. A half-processed\n'
                  '         dataset teaches the model the processing, not the disease.\n'
                  '         These are probably whole-plant photos - set\n'
                  '         REMOVE_BACKGROUND = False and rerun from Cell 1.')
    print('\n' + '=' * 25, s, '=' * 25)
    r = subprocess.run([sys.executable, s])
    assert r.returncode == 0, s + ' FAILED - read its output above before rerunning'
print('\npipeline data stages done')

In [ ]:
# Cell 4 - train the ConvNeXt classifier (contender B recipe, seed 42). ~30-60 min on T4.
r = subprocess.run([sys.executable, 'src/train.py', '--mode', 'B'])
assert r.returncode == 0 and Path('models/v1/contender_B.pt').exists(), 'training failed'
print('training done -> models/v1/contender_B.pt (renamed at packaging time)')

In [ ]:
# Cell 5 - thresholds, derived from the HEALTHY validation scores
# (never from infected quantiles - that was the documented v1 bug)
import json, sys, numpy as np, timm
sys.path.insert(0, '/content/cwsb/src')
import train as Tr

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ck = torch.load('models/v1/contender_B.pt', map_location=device, weights_only=False)
new_model = timm.create_model(ck['backbone'], pretrained=False, num_classes=2).to(device)
new_model.load_state_dict(ck['state_dict'])
new_model.eval()
print(f"new model: epoch {ck['epoch']}  val PR-AUC {float(ck['val_pr_auc']):.4f}")

va = Tr.load_rows('val')
yv, pv = Tr.scores(new_model, va, device)
healthy_val, infected_val = pv[yv == 0], pv[yv == 1]
print(f'val healthy  n={len(healthy_val)}  scores {healthy_val.min():.3f} - {healthy_val.max():.3f}'
      if len(healthy_val) else 'val healthy: NONE')
print(f'val infected n={len(infected_val)}  scores {infected_val.min():.3f} - {infected_val.max():.3f}')

T_HIGH = 0.90
if len(healthy_val) >= 5:
    T_LOW = float(np.clip(round(float(healthy_val.max()) + 0.05, 2), 0.35, 0.60))
    basis = f'max healthy val score {healthy_val.max():.3f} + 0.05 margin, clamped to [0.35, 0.60]'
else:
    T_LOW = 0.50
    basis = f'DEFAULT - only {len(healthy_val)} healthy val photos, too few to calibrate on'
    print('WARNING:', basis)
print(f'T_low = {T_LOW}   T_high = {T_HIGH}   ({basis})')

config = {
    'model': f'{TAG}.pt', 'backbone': ck['backbone'], 'img': ck['img'],
    # serving reads these two and applies the identical step - do not edit by hand
    'remove_bg': bool(REMOVE_BACKGROUND), 'remove_bg_fill': BG_FILL,
    'version': VERSION_NUMBER, 'T_low': T_LOW, 'T_high': T_HIGH,
    'plant_score': "max over the plant's photos",
    'answers': {'score >= T_high': 'wsb_likely',
                'T_low <= score < T_high': 'inconclusive',
                'score < T_low': 'no_visible_signs'},
    'threshold_basis': basis,
    'calibrated_on': 'validation split only; test split used as a check',
}
Path('models/v1/config.json').write_text(json.dumps(config, indent=2))
print('-> models/v1/config.json')

In [ ]:
# Cell 6 - held-out TEST evaluation: new model vs old model, same photos, same thresholds
if REMOVE_BACKGROUND:
    print('NOTE: the OLD model was trained on full frames but is being scored on\n'
          '      background-removed photos here, so its numbers are unfairly low.\n'
          '      Judge the new model on its own recall/clears, not on this gap.')
te = Tr.load_rows('test')
yt, pt_new = Tr.scores(new_model, te, device)

def report(name, y, p):
    v = np.where(p >= T_HIGH, 'wsb', np.where(p < T_LOW, 'clear', 'inc'))
    caught = int(((y == 1) & (v == 'wsb')).sum());   missed = int(((y == 1) & (v == 'clear')).sum())
    fa = int(((y == 0) & (v == 'wsb')).sum());       cleared = int(((y == 0) & (v == 'clear')).sum())
    inc = int((v == 'inc').sum())
    rec = caught / max(caught + missed, 1)
    print(f'{name:22s} n={len(y):3d}  caught {caught:3d}  missed {missed:2d}  '
          f'false-alarms {fa:2d}  cleared {cleared:3d}  inconclusive {inc:3d}  recall {rec:.1%}')

print(f'TEST split ({len(te)} photos, thresholds {T_LOW}/{T_HIGH}):')
report(f'NEW  {TAG}', yt, pt_new)

if OLD_PT:
    cko = torch.load(OLD_PT, map_location=device, weights_only=False)
    sd = cko.get('state_dict', cko) if isinstance(cko, dict) else cko
    old_model = timm.create_model(cko.get('backbone', ck['backbone']) if isinstance(cko, dict) else ck['backbone'],
                                  pretrained=False, num_classes=2).to(device)
    old_model.load_state_dict({k.replace('module.', ''): v for k, v in sd.items()})
    old_model.eval()
    _, pt_old = Tr.scores(old_model, te, device)
    report('OLD  ' + Path(OLD_PT).name, yt, pt_old)
    print('\nNote: the old model also sees the NEW test photos here - a fair same-data comparison.')
    del old_model
else:
    print('(no old_models.zip provided - skipping the baseline comparison)')

In [ ]:
# Cell 7 - ONNX export + 4-decimal parity check
import torch.nn as nn, torch.nn.functional as F
import onnxruntime as ort
from PIL import Image
from torchvision import transforms

IMG = ck['img']
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

class CwsbOnnx(nn.Module):
    """Input: 384x384x3 RGB uint8 (already resized+center-cropped outside the graph).
    Output: probabilities (1,2)=[healthy,wsb] and feature_map (1,768,12,12) for heatmaps."""
    def __init__(self, core):
        super().__init__()
        self.core = core
        self.register_buffer('mean', torch.tensor(MEAN).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(STD).view(1, 3, 1, 1))
    def forward(self, rgb_uint8):
        x = rgb_uint8.permute(2, 0, 1).unsqueeze(0).float() / 255.0
        x = (x - self.mean) / self.std
        feat = self.core.forward_features(x)
        logits = self.core.forward_head(feat)
        return F.softmax(logits, dim=1), feat

cpu_model = new_model.to('cpu').eval()
onnx_path = f'models/v1/{TAG}.onnx'
torch.onnx.export(CwsbOnnx(cpu_model).eval(),
                  torch.randint(0, 256, (IMG, IMG, 3), dtype=torch.uint8), onnx_path,
                  input_names=['rgb_384'], output_names=['probabilities', 'feature_map'],
                  opset_version=17, dynamo=False)

crop_tf = transforms.Compose([transforms.Resize(IMG + 32), transforms.CenterCrop(IMG)])
norm_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
worst = 0.0
for _ in range(5):
    h, w = np.random.randint(500, 3000), np.random.randint(500, 3000)
    crop = np.asarray(crop_tf(Image.fromarray(
        np.random.randint(0, 256, (h, w, 3), dtype=np.uint8))), dtype=np.uint8)
    with torch.no_grad():
        ref = torch.softmax(cpu_model(norm_tf(Image.fromarray(crop)).unsqueeze(0)), 1).numpy()
    got = sess.run(['probabilities'], {'rgb_384': crop})[0]
    worst = max(worst, float(np.max(np.abs(ref - got))))
print(f'max |pytorch - onnx| over 5 inputs: {worst:.2e}')
assert worst < 1e-4, f'parity FAILED ({worst})'
print(f'Parity PASSED -> {onnx_path}  ({os.path.getsize(onnx_path)/1e6:.1f} MB)')

In [ ]:
# Cell 8 - package everything as the versioned release and download
from google.colab import files as gfiles
rel = Path(f'/content/{TAG}_release')
shutil.rmtree(rel, ignore_errors=True)
rel.mkdir()
shutil.copy('models/v1/contender_B.pt', rel / f'{TAG}.pt')
shutil.copy(onnx_path, rel / f'{TAG}.onnx')
shutil.copy('models/v1/config.json', rel / f'config_v{VERSION_NUMBER}.json')
shutil.copy('models/v1/training_result.json', rel / f'training_result_v{VERSION_NUMBER}.json')
shutil.copy('work/splits/split.csv', rel / f'split_v{VERSION_NUMBER}.csv')

zip_name = f'{TAG}_release.zip'
with zipfile.ZipFile('/content/' + zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in rel.iterdir():
        zf.write(p, arcname=p.name)
print(f'{zip_name}: ' + ', '.join(p.name for p in sorted(rel.iterdir())))
gfiles.download('/content/' + zip_name)

## After downloading — on the local machine

1. Unzip **flat** into `models/v{N}/` (e.g. `models/v2/`) — `cwsb_v2.pt` directly inside, no
   nested folders (known Colab unzip pitfall). **Never overwrite the previous version's folder.**
2. Decide whether vN actually ships: compare the old-vs-new table from Cell 6, and re-run the
   local checks (`test2/`, `test_new/`) against the new `.pt`. A retrain can be worse — the old
   model stays the production one until the new one wins.
3. Copy `split_v{N}.csv` into `work/splits/` so the split is reproducible.
4. Log the run (data counts, thresholds, test table, parity number) in `charan_working.md`
   and update `PROJECT_STATE.md` / `NEXT_STEPS.md`.

**Reminders about the data going in:** new *distinct plants* beat more photos of known plants;
whole-plant 2–3 m framing only (close-ups belong to a different model); labels verified by stem
inspection. The healthy class is the bottleneck — prioritise it.